# Basic Deployment Operations `@azure/ai-projects`

This notebook demonstrates how to use the `AIProjectClient` to manage deployments: list all deployments, list deployments filtered by model publisher, and get a single deployment by name.

It mirrors the [`deploymentsBasics.ts`](./deploymentsBasics.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `MODEL_PUBLISHER`.

Run the cells in order (top to bottom); state is shared across cells.

In [ ]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";
import type { ModelDeployment } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";


In [10]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [11]:
// List all deployments
console.log("List all deployments:");
const deployments: ModelDeployment[] = [];
const properties: Array<Record<string, string>> = [];

const listDeployments = async () => {
  for await (const deployment of project.deployments.list()) {
    // Check if this is a ModelDeployment (has the required properties)
    if (
      deployment.type === "ModelDeployment" &&
      "modelName" in deployment &&
      "modelPublisher" in deployment &&
      "modelVersion" in deployment
    ) {
      deployments.push(deployment);
      properties.push({
        name: deployment.name,
        modelPublisher: deployment.modelPublisher,
        modelName: deployment.modelName,
      });
    }
  }
};
await listDeployments();
console.log(`Retrieved deployments: ${JSON.stringify(properties, null, 2)}`);

List all deployments:
Retrieved deployments: [
  {
    "name": "gpt-4o",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-4o"
  },
  {
    "name": "gpt-5",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-5"
  },
  {
    "name": "computer-use-preview",
    "modelPublisher": "OpenAI",
    "modelName": "computer-use-preview"
  },
  {
    "name": "gpt-5.2",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-5.2"
  },
  {
    "name": "gpt-image-1",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-image-1.5"
  },
  {
    "name": "gpt-4o-mini",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-4o-mini"
  },
  {
    "name": "text-embedding-3-large",
    "modelPublisher": "OpenAI",
    "modelName": "text-embedding-3-large"
  },
  {
    "name": "gpt-5.2-chat",
    "modelPublisher": "OpenAI",
    "modelName": "gpt-5.3-chat"
  },
  {
    "name": "text-embedding-3-small-1",
    "modelPublisher": "OpenAI",
    "modelName": "text-embedding-3-small"
  },
  {
    "name": "gpt-5.6

In [14]:
// List all deployments by a specific model publisher
console.log(`List all deployments by the model publisher 'OpenAI':`);
const filteredDeployments: ModelDeployment[] = [];
const listByPublisher = async () => {
  for await (const deployment of project.deployments.list({
    modelPublisher: "OpenAI",
  })) {
    // Check if this is a ModelDeployment
    if (
      deployment.type === "ModelDeployment" &&
      "modelName" in deployment &&
      "modelPublisher" in deployment &&
      "modelVersion" in deployment
    ) {
      filteredDeployments.push(deployment);
    }
  }
};
await listByPublisher();
console.log(
  `Retrieved ${filteredDeployments.length} deployments from model publisher '${modelPublisher}'`,
);

List all deployments by the model publisher 'OpenAI':
Retrieved 10 deployments from model publisher 'Microsoft'


In [15]:
// Get a single deployment by name
if (deployments.length > 0) {
  const deploymentName = deployments[0].name;
  console.log(`Get a single deployment named '${deploymentName}':`);
  const singleDeployment = await project.deployments.get(deploymentName);
  console.log(`Retrieved deployment: ${JSON.stringify(singleDeployment, null, 2)}`);
} else {
  console.log("No deployments available to retrieve.");
}

Get a single deployment named 'gpt-4o':
Retrieved deployment: {
  "type": "ModelDeployment",
  "name": "gpt-4o",
  "modelName": "gpt-4o",
  "modelVersion": "2024-11-20",
  "modelPublisher": "OpenAI",
  "capabilities": {
    "chat_completion": "true",
    "completion": "true"
  },
  "sku": {
    "capacity": 308,
    "name": "GlobalStandard"
  }
}
